In [ ]:
%load_ext autoreload
%autoreload 2

### 1. 모듈 임포트

In [ ]:
import torch
import numpy as np
# 1. 모듈 임포트
from modules.config import config
from modules.sdf_generator import SDFGenerator
from modules.particle_sampler import sample_particles_poisson
from modules.sdf_network import FeatureConstruction
from modules.visualizer import visualize_simulation, visualize_particles_and_features


In [ ]:
# ==========================================
# Setup: Config & 디바이스 초기화
# ==========================================
print(f"⚙️ Config 설정: Domain Size={config.domain_size}, Resolution={config.resolution}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"⚡ 디바이스 초기화: {device}")

In [ ]:
import torch_directml
device = torch_directml.device() 
print(f"현재 디바이스: {device}")


## 2. 데이터셋 생성

### 2.1 데이터 셋 sdf 생성 및 Poisson Disk 샘플링

In [ ]:
import os
import glob

OUTPUT_DIR = "dataset"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Clean up old single shape files to avoid confusion
old_files = glob.glob(os.path.join(OUTPUT_DIR, "*.npy"))
for f in old_files:
    os.remove(f)
    print(f"Removed old dataset file: {f}")

DATASET_SIZE = 5

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
generator = SDFGenerator(config)
feature_constructor = FeatureConstruction(dx=config.dx, device=device)

print(f"\n🚀 Generating {DATASET_SIZE} training samples (SDF + m_c grids)...")

for i in range(DATASET_SIZE):
    print(f"[{i+1}/{DATASET_SIZE}] Generating shape...")
    
    # 1. Generate SDF
    random_shape = generator.create_random_shape(seed=42 + i)
    sdf_grid = generator.to_grid(random_shape)
    
    # 2. Sample Particles
    print("  Sampling particles...")
    particles = sample_particles_poisson(sdf_grid, config)
    
    # 3. Calculate m_c feature grid
    print("  Calculating m_c features...")
    particles_tensor = torch.tensor(particles, dtype=torch.float32, device=device)
    grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
    mc_grid = m_c.reshape(grid_shape).cpu().numpy()
    
    # 4. Save
    sdf_filename = os.path.join(OUTPUT_DIR, f"sdf_grid_{i:03d}.npy")
    mc_filename = os.path.join(OUTPUT_DIR, f"mc_grid_{i:03d}.npy")
    
    np.save(sdf_filename, sdf_grid)
    np.save(mc_filename, mc_grid)
    print(f"  Saved: {sdf_filename}, {mc_filename}")

print("\n✅ Dataset generation complete! Files are ready in the 'dataset' folder.")


In [ ]:
import numpy as np
import torch
from modules.config import config
from modules.visualizer import visualize_simulation, visualize_feature_grid

# 1. 확인할 데이터 인덱스 설정 (0 ~ 4)
sample_idx = 1
sdf_file = f"dataset/sdf_grid_{sample_idx:03d}.npy"
mc_file = f"dataset/mc_grid_{sample_idx:03d}.npy"

# 2. 데이터 로드 완료
sdf_grid = np.load(sdf_file)
mc_grid = np.load(mc_file)
print(f"📥 SDF 데이터 로드: {sdf_file} (Shape: {sdf_grid.shape})")
print(f"📥 m_c 데이터 로드: {mc_file} (Shape: {mc_grid.shape})")

# 3. [시각화 1] Target SDF (원본 3D 지오메트리 형태)
print("\n[1] 정답(Target) SDF 형상 시각화")
visualize_simulation(
    sdf_grid=sdf_grid, 
    domain_size=config.domain_size, 
    title=f"Target SDF Shape (Sample {sample_idx})"
)

In [ ]:

# 4. [시각화 2] Input 3D CNN Features (파티클로부터 밀도 단위로 추출된 m_c 값)
print("\n[2] 입력(Input) m_c 특징 공간 시각화")

fig2 = visualize_feature_grid(
    m_c_grid=mc_grid,         # 앞서 파이프라인에서 나온 mc_grid 텐서 그대로 삽입
    grid_nodes=grid_nodes,    # 앞서 파이프라인에서 나온 grid_nodes 텐서 그대로 삽입
    title=f"Input m_c Feature Grid (Sample {sample_idx})"
)

## 3. sdf 계산 -CNN 학습

### 3.1 전처리 단계: 그리드 특징값(m_c) 계산

### 3.2 m_c 값 시각화

In [ ]:
print("\n=== [Step 3] 시각화 1: 그리드 특징값(m_c) ===\n")

# 파티클과 그리드 특징값 동시 시각화
fig1 = visualize_particles_and_features(
    particles=particles,
    grid_nodes=grid_nodes,
    m_c_grid=m_c.reshape(grid_shape),
    title="particle(red) and grid feature value(colormapped)"
)

# 그리드 특징값만 시각화
fig2 = visualize_feature_grid(
    m_c_grid=m_c.reshape(grid_shape),
    grid_nodes=grid_nodes,
    title="grid feature value m_c (Poly6 kernel based)"
)

print("✓ m_c 시각화 완료")

In [ ]:
import torch
import numpy as np

# 1. 모듈 임포트
from modules.config import config
from modules.sdf_generator import SDFGenerator
from modules.particle_sampler import sample_particles_poisson
from modules.sdf_network import FeatureConstruction
from modules.visualizer import visualize_simulation, visualize_particles_and_features

# ==========================================
# Setup: Config & 디바이스 초기화
# ==========================================
print(f"⚙️ Config 설정: Domain Size={config.domain_size}, Resolution={config.resolution}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ==========================================
# Phase 1: SDF 도형 생성 및 래스터라이징
# ==========================================
print("\n[Phase 1] 🎲 랜덤 SDF 도형 생성 중...")
# 새로 만든 OOP 클래스를 사용합니다.
generator = SDFGenerator(config)

random_sdf = generator.create_random_shape(seed=2024) 
sdf_grid = generator.to_grid(random_sdf)

print(f"✅ SDF Grid 생성 완료: Shape={sdf_grid.shape}")

# ==========================================
# Phase 2: Poisson Disk 샘플링
# ==========================================
print("\n[Phase 2] 🎯 파티클 샘플링 중...")
# config 객체를 두 번째 인자로 넘겨받도록 변경된 부분을 반영합니다.
particles = sample_particles_poisson(sdf_grid, config)

print(f"✅ 파티클 샘플링 완료: {len(particles)}개 생성됨 (목표: {config.num_particles}개)")

# ==========================================
# Phase 3: 특징 인코딩 (Feature Construction)
# ==========================================
print("\n[Phase 3] 🧠 모델 입력용 특징(m_c) 추출 중...")
# 함수형 형태를 유지한 원본 FeatureConstruction을 사용하되, dx는 config 참조
feature_constructor = FeatureConstruction(dx=config.dx, device=device)

particles_tensor = torch.tensor(particles, dtype=torch.float32, device=device)

# 파티클 위치 기반으로 동적 그리드 변환 
grid_nodes, m_c, grid_shape = feature_constructor(particles_tensor)
m_c_grid = m_c.reshape(grid_shape).cpu().numpy()

print(f"✅ 기하학적 특징 추출 완료")
print(f"   - Grid Shape: {grid_shape}")
print(f"   - m_c 통계: Min={m_c.min():.4f}, Max={m_c.max():.4f}")



In [ ]:

# ==========================================
# Phase 4: 시각화 모듈
# ==========================================
print("\n[Phase 4] 📊 프로세스 검증용 데이터 시각화")

# 4-1. 전반적인 형상 + 파티클 위치 뷰어
# visualize_simulation(
#     sdf_grid=sdf_grid, 
#     particles=particles, 
#     domain_size=config.domain_size, 
#     title="SDF Model & Generated Particles"
# )

# 4-2. 파티클 분포와 네트워크의 m_c 추출 결과 뷰어
visualize_particles_and_features(
    particles=particles_tensor.cpu(),
    grid_nodes=grid_nodes.cpu(),
    m_c_grid=m_c_grid,
    title="Network Feature Distribution (m_c)"
)

### 3.3 3D CNN 네트워크를 통한 SDF 값 추론

### 3.4 학습

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch_directml
from torch.utils.data import DataLoader

from modules.dataset import SDFDataset
from modules.sdf_network import SDFNetwork

print("=== [Step 1 & 2] 디바이스, 데이터셋 및 모델 초기화 ===")

# 1. 🚀 DirectML 가속 디바이스 할당 (AMD GPU 가동!)
device = torch_directml.device()
print(f"🔥 현재 사용 중인 가속 디바이스: {device}")

# 2. 📦 학습 데이터셋(5개 기본 도형) 메모리 로드
print("\n[데이터셋 로드 중...]")
train_dataset = SDFDataset(data_dir="dataset", patch_size=8, in_memory=True)
# 배치 사이즈: 512 (GPU 메모리 상태에 따라 추후 가감 가능)
train_loader = DataLoader(train_dataset, batch_size=512, shuffle=True, drop_last=True)
print(f"✅ 총 {len(train_dataset)} 개의 (8x8x8) 학습 패치 추출 완료!")
print(f"✅ 1 에폭(Epoch)당 미니배치 개수: {len(train_loader)} 번")

# 3. 🧠 3D CNN 모델 인스턴스화
model = SDFNetwork().to(device)

# 4. 📉 손실 함수 및 최적화 알고리즘
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

print("\n✅ 모델, Loss, Optimizer까지 완벽하게 스탠바이가 완료되었습니다!")


In [ ]:
# SDFNetwork 학습 예제
import torch
from torch.utils.data import DataLoader, TensorDataset
from modules.sdf_network import SDFNetwork

# 예시 데이터 준비 (실제 프로젝트에서는 m_c_patch, sdf_gt를 생성해야 함)
# 아래는 데모용 랜덤 데이터
batch_size = 16
num_samples = 128
patch_shape = (1, 8, 8, 8)

# 랜덤 feature patch와 SDF GT 생성
feature_patches = torch.randn(num_samples, *patch_shape)
sdf_gt = torch.randn(num_samples)

dataset = TensorDataset(feature_patches, sdf_gt)
train_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

# 네트워크, 옵티마이저, 손실함수 정의
model = SDFNetwork().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = torch.nn.MSELoss()

# 학습 실행
num_epochs = 5
losses = model.train_step(train_loader, optimizer, criterion, device=device, num_epochs=num_epochs)

print("학습 완료. 에폭별 손실:", losses)
# 학습된 모델로 추론 예시
# (여기서는 랜덤 feature patch 중 일부로 추론)
model.eval()
with torch.no_grad():
    test_patch = feature_patches[0:1].to(device)  # 첫 번째 패치
    pred_sdf = model(test_patch)
    print(f"추론 결과 SDF 값: {pred_sdf.item():.6f}")

In [ ]:
import torch.nn as nn
from torch.optim import Adam
from modules.dataset import create_dataloader

# 1. Sliding Window 기반 학습 데이터 로더 생성
print("[준비] 학습 데이터 로드 중...")
train_loader = create_dataloader(data_dir="dataset", batch_size=64, in_memory=True)

# 2. 손실 함수 (MSE) 및 옵티마이저 (Adam) 설정
criterion = nn.MSELoss()
optimizer = Adam(reconstruction.network.parameters(), lr=1e-4)

# 3. 모델 학습 루프 (FeatureConstruction은 제외하고 network만 훈련)
print("[학습] 3D CNN SDF 추론 학습 시작")
train_losses = reconstruction.network.train_step(
    train_loader=train_loader, 
    optimizer=optimizer, 
    criterion=criterion, 
    device=device, 
    num_epochs=5, 
    verbose=True
)
print("[학습] 완료!")

## 4. SDF 값 시각화 (최종 결과)

4.2 obj로 저장(mc 알고리즘)

In [ ]:
from modules.visualizer import save_mesh_as_obj # 시각화용
save_mesh_as_obj(sdf_grid, filename="test_shape_64.obj")


npy 파일 이용한 sdf 시각화

In [ ]:
from modules.visualizer import visualize_npy
# 2. 데이터 시각화 (Phase 2)
print("\n--- Phase 2: 시각화 및 저장 ---")

# SDF 파일 확인
visualize_npy("sdf_grid_64.npy")

# 파티클 파일 확인
visualize_npy("particles.npy")


변수(객체) 이용한 시각화(런타임에 있는 메모리)

In [ ]:
from modules.visualizer import visualize_simulation

# SDF만 보기 (기존 plot_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid)

# 파티클까지 겹쳐서 보기 (기존 plot_particles_and_sdf_slice + 3D)
visualize_simulation(sdf_grid=sdf_grid, particles=particles)

obj 인터랙티브 뷰어

In [ ]:
from modules.visualizer import view_obj_interactive
# OBJ 파일 인터랙티브 뷰어
view_obj_interactive("test_shape_64.obj")